In [1]:
!pip install yfinance openpyxl plotly --quiet

# Módulo 4: Panel de Comparación Cruzada de Estrategias

Este notebook consolida y evalúa las 7 estrategias comparativas bajo un único motor de backtesting justo. Se realiza la simulación histórica del crecimiento del capital considerando costos transaccionales, rebalanceos y la política óptima calculada mediante Programación Dinámica. Finalmente, se computan métricas históricas de riesgo y retorno, se generan rankings de desempeño y se exporta un reporte detallado en Excel.

In [2]:
import os
import json
import io
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px

# Crear carpeta de salidas si no existe
os.makedirs("salidas", exist_ok=True)

# 1. Carga robusta de JSONs de los módulos previos
res_m1 = {}
res_m2 = {}
res_m3 = {}

if os.path.exists("resultados_m1.json"):
    with open("resultados_m1.json", "r", encoding="utf-8") as f:
        res_m1 = json.load(f)
if os.path.exists("resultados_m2.json"):
    with open("resultados_m2.json", "r", encoding="utf-8") as f:
        res_m2 = json.load(f)
if os.path.exists("resultados_m3.json"):
    with open("resultados_m3.json", "r", encoding="utf-8") as f:
        res_m3 = json.load(f)

# Extraer parámetros con fallbacks en caso de que falten los JSONs
TICKERS = res_m1.get("tickers", ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM'])
config_m1 = res_m1.get("configuracion", {})
FECHA_INICIO = config_m1.get("fecha_inicio", '2015-01-01')
FECHA_FIN = config_m1.get("fecha_fin", '2024-12-31')
CAPITAL_INICIAL = float(config_m1.get("capital_inicial", 100000))
frecuencia_sel = config_m1.get("frecuencia", 'Mensual')
DIAS_ANIO = int(config_m1.get("dias_anio", 252))
RF = float(config_m1.get("rf", 0.0))

params_dp = res_m3.get("parametros_dp", {})
LAMBDA_TC = float(params_dp.get("lambda_tc", 0.0010))
PASO_GRILLA = float(params_dp.get("paso_grilla", 0.10))
T_DP = int(params_dp.get("periodos_t", 12))

# Cargar grilla y política de M3 para rebalanceo DP real
S_matriz = np.array(res_m3.get("grilla_estados", []), dtype=float)
politica_matriz = np.array(res_m3.get("matriz_politica", []), dtype=int)

print("Configuración cargada con éxito:")
print(f"  Tickers: {TICKERS}")
print(f"  Capital Inicial: ${CAPITAL_INICIAL:,.2f}")
print(f"  Frecuencia: {frecuencia_sel}")
print(f"  Costo transaccional: {LAMBDA_TC}")

Configuración cargada con éxito:
  Tickers: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']
  Capital Inicial: $100,000.00
  Frecuencia: Mensual
  Costo transaccional: 0.001


## 1. Extracción de Pesos Óptimos

Cargamos los pesos óptimos de Markowitz (Módulo 1) y NSGA-II (Módulo 2). Si no están disponibles, hacemos un fallback a un portafolio equiponderado.

In [3]:
def extraer_pesos(data, clave_estrategia):
    if clave_estrategia in data and "pesos" in data[clave_estrategia]:
        pesos_dict = data[clave_estrategia]["pesos"]
        pesos = np.array([float(pesos_dict.get(t, 0.0)) for t in TICKERS], dtype=float)
        if pesos.sum() > 0:
            return pesos / pesos.sum()
    return np.ones(len(TICKERS)) / len(TICKERS)

w_markowitz = extraer_pesos(res_m1, "markowitz_max_sharpe")
w_nsga2 = extraer_pesos(res_m2, "nsga2_max_sharpe")
w_eq = np.ones(len(TICKERS)) / len(TICKERS)
w_target_dp = w_markowitz.copy()

print("Pesos Markowitz:", dict(zip(TICKERS, w_markowitz.round(4))))
print("Pesos NSGA-II:", dict(zip(TICKERS, w_nsga2.round(4))))

Pesos Markowitz: {'ABX.TO': np.float64(0.3773), 'BHP': np.float64(0.6227), 'BVN': np.float64(0.0), 'FSM': np.float64(0.0), 'VOLCABC1.LM': np.float64(0.0)}
Pesos NSGA-II: {'ABX.TO': np.float64(0.3594), 'BHP': np.float64(0.64), 'BVN': np.float64(0.0002), 'FSM': np.float64(0.0002), 'VOLCABC1.LM': np.float64(0.0001)}


## 2. Descarga de Precios y Procesamiento de Retornos

Descargamos el histórico unificado para garantizar que la comparación sea justa, aplicando forward-fill e interpolación y remuestreando los precios a la frecuencia seleccionada.

In [4]:
print("Descargando datos históricos unificados...")
descarga = yf.download(TICKERS, start=FECHA_INICIO, end=FECHA_FIN, progress=False)
if isinstance(descarga, pd.DataFrame) and not descarga.empty:
    precios = descarga['Close'] if 'Close' in descarga.columns else descarga
else:
    raise ValueError("Error al descargar precios de Yahoo Finance.")

precios = precios.ffill().bfill()
TICKERS_ORDENADOS = list(precios.columns)

# Realinear todos los pesos a las columnas del DataFrame de precios
w_markowitz = np.array([dict(zip(TICKERS, w_markowitz)).get(t, 1/len(TICKERS_ORDENADOS)) for t in TICKERS_ORDENADOS], dtype=float)
w_markowitz /= w_markowitz.sum()

w_nsga2 = np.array([dict(zip(TICKERS, w_nsga2)).get(t, 1/len(TICKERS_ORDENADOS)) for t in TICKERS_ORDENADOS], dtype=float)
w_nsga2 /= w_nsga2.sum()

w_eq = np.ones(len(TICKERS_ORDENADOS)) / len(TICKERS_ORDENADOS)
w_target_dp = w_markowitz.copy()

# Resampleo mensual
mapa_freq = {"Semanal": "W-FRI", "Mensual": "ME", "Trimestral": "QE"}
codigo_freq = mapa_freq.get(frecuencia_sel, "ME")
precios_res = precios.resample(codigo_freq).last()
retornos_periodo = precios_res.pct_change().dropna()
T = len(retornos_periodo)

fechas_sim = [precios_res.index[0]] + list(retornos_periodo.index)

print(f"Periodos de backtest: {T} ({frecuencia_sel})")

Descargando datos históricos unificados...


Periodos de backtest: 119 (Mensual)


## 3. Motor Único de Backtesting

Definimos la función unificada de simulación que maneja las tres modalidades de ejecución: Buy & Hold (B&H), Rebalanceo Periódico Estricto, y Rebalanceo por Programación Dinámica (DP).

In [5]:
def backtest_estrategia(retornos_df, w_objetivo, capital_inicial,
                        modo="buy_hold", lambda_tc=0.0, w_target_dp=None,
                        S_dp=None, politica_dp=None, T_dp=12):
    R = retornos_df.to_numpy(dtype=float)
    T_len, N_len = R.shape
    w_objetivo = np.asarray(w_objetivo, dtype=float)
    if w_target_dp is None:
        w_target_dp = w_objetivo.copy()

    riqueza = np.zeros(T_len + 1)
    riqueza[0] = capital_inicial
    w_actual = w_objetivo.copy()
    rebalanceos = []
    costo_total = 0.0

    for t in range(T_len):
        ret = R[t]

        if modo == "rebalancear" and t > 0:
            delta = np.sum(np.abs(w_actual - w_objetivo))
            tc = lambda_tc * delta
            costo_total += tc * riqueza[t]
            cap_post = riqueza[t] * (1 - tc)
            ret_port = float(np.dot(w_objetivo, ret))
            riqueza[t + 1] = cap_post * (1 + ret_port)
            w_actual = w_objetivo * (1 + ret) / (1 + ret_port)
            rebalanceos.append(t)

        elif modo == "dp" and t > 0:
            if S_dp is not None and politica_dp is not None and S_dp.size > 0:
                idx_s = int(np.argmin(np.sum(np.abs(S_dp - w_actual), axis=1)))
                idx_a = int(politica_dp[min(t, T_dp - 1), idx_s])
                w_dp_nuevo = S_dp[idx_a]
                
                if idx_s != idx_a:
                    tc = lambda_tc * np.sum(np.abs(w_actual - w_dp_nuevo))
                    costo_total += tc * riqueza[t]
                    cap_post = riqueza[t] * (1 - tc)
                    ret_port = float(np.dot(w_dp_nuevo, ret))
                    riqueza[t + 1] = cap_post * (1 + ret_port)
                    w_actual = w_dp_nuevo * (1 + ret) / (1 + ret_port)
                    rebalanceos.append(t)
                else:
                    ret_port = float(np.dot(w_actual, ret))
                    riqueza[t + 1] = riqueza[t] * (1 + ret_port)
                    w_actual = w_actual * (1 + ret) / (1 + ret_port)
            else:
                # Fallback simple en caso de no haber grilla
                desv = np.sum(np.abs(w_actual - w_target_dp))
                if desv > 0.15:
                    tc = lambda_tc * desv
                    costo_total += tc * riqueza[t]
                    cap_post = riqueza[t] * (1 - tc)
                    ret_port = float(np.dot(w_target_dp, ret))
                    riqueza[t + 1] = cap_post * (1 + ret_port)
                    w_actual = w_target_dp * (1 + ret) / (1 + ret_port)
                    rebalanceos.append(t)
                else:
                    ret_port = float(np.dot(w_actual, ret))
                    riqueza[t + 1] = riqueza[t] * (1 + ret_port)
                    w_actual = w_actual * (1 + ret) / (1 + ret_port)

        else:
            ret_port = float(np.dot(w_actual, ret))
            riqueza[t + 1] = riqueza[t] * (1 + ret_port)
            w_actual = w_actual * (1 + ret) / (1 + ret_port)

    return riqueza, rebalanceos, costo_total

## 4. Ejecución del Backtesting de las 7 Estrategias

Corremos las simulaciones de riqueza para las 7 combinaciones de asignaciones (Equiponderado, Markowitz, NSGA-II) y rebalanceos (B&H, Rebalanceado, DP).

In [6]:
estrategias = [
    ("1. Equiponderado (Buy & Hold)",                       "buy_hold",    w_eq,        0.0),
    (f"2. Equiponderado (Rebalanceado {frecuencia_sel})",   "rebalancear", w_eq,        LAMBDA_TC),
    ("3. Markowitz Máx. Sharpe (Buy & Hold)",               "buy_hold",    w_markowitz, 0.0),
    (f"4. Markowitz (Rebalanceado {frecuencia_sel})",       "rebalancear", w_markowitz, LAMBDA_TC),
    ("5. NSGA-II (Buy & Hold)",                             "buy_hold",    w_nsga2,     0.0),
    (f"6. NSGA-II (Rebalanceado {frecuencia_sel})",         "rebalancear", w_nsga2,     LAMBDA_TC),
    ("7. Programación Dinámica (Bellman)",                  "dp",          w_target_dp, LAMBDA_TC),
]

trayectorias = {}
rebalanceos_dict = {}
costos_dict = {}

for nombre, modo, w, lam in estrategias:
    w_path, reb, cost = backtest_estrategia(
        retornos_periodo, w, CAPITAL_INICIAL, modo, lam, w_target_dp,
        S_matriz, politica_matriz, T_DP
    )
    trayectorias[nombre] = w_path
    rebalanceos_dict[nombre] = reb
    costos_dict[nombre] = cost

nombres_estrategias = [e[0] for e in estrategias]
W = np.vstack([trayectorias[n] for n in nombres_estrategias])
print("Backtests completados exitosamente para las 7 estrategias.")

Backtests completados exitosamente para las 7 estrategias.


## 5. Cálculo de Métricas y Rankings

Calculamos métricas anualizadas de retorno total, Sharpe Ratio, Sortino Ratio, volatilidad y drawdowns máximos de cada una de las estrategias simuladas.

In [7]:
factor_anual = {"Semanal": 52, "Mensual": 12, "Trimestral": 4}.get(frecuencia_sel, 12)

def calcular_metricas(riqueza_path, factor_anual, rf=0.0):
    riqueza_path = np.asarray(riqueza_path, dtype=float)
    retornos = np.diff(riqueza_path) / riqueza_path[:-1]
    if len(retornos) < 2:
        return {
            "Riqueza Final ($)": 0.0, "Retorno Total (%)": 0.0,
            "Sharpe Ratio": 0.0, "Sortino Ratio": 0.0,
            "Max Drawdown (%)": 0.0, "Volatilidad Anual (%)": 0.0
        }

    ret_total = (riqueza_path[-1] / riqueza_path[0] - 1) * 100
    mu_p = np.mean(retornos)
    sd_p = np.std(retornos, ddof=1)
    mu_anual = mu_p * factor_anual
    sig_anual = sd_p * np.sqrt(factor_anual)
    sharpe = (mu_anual - rf) / sig_anual if sig_anual > 0 else 0.0

    downside = retornos[retornos < 0]
    if len(downside) > 1:
        dd_std = np.std(downside, ddof=1) * np.sqrt(factor_anual)
        sortino = (mu_anual - rf) / dd_std if dd_std > 0 else 0.0
    else:
        sortino = 0.0

    cum_max = np.maximum.accumulate(riqueza_path)
    drawdowns = (riqueza_path - cum_max) / cum_max
    max_dd = drawdowns.min() * 100

    return {
        "Riqueza Final ($)": float(riqueza_path[-1]),
        "Retorno Total (%)": float(ret_total),
        "Sharpe Ratio": float(sharpe),
        "Sortino Ratio": float(sortino),
        "Max Drawdown (%)": float(max_dd),
        "Volatilidad Anual (%)": float(sig_anual * 100),
    }

metricas_lista = []
for nombre, *_ in estrategias:
    m = calcular_metricas(trayectorias[nombre], factor_anual, RF)
    m["Estrategia"] = nombre
    m["Costo Total TC ($)"] = float(costos_dict[nombre])
    m["N° Rebalanceos"] = len(rebalanceos_dict[nombre])
    metricas_lista.append(m)

columnas_orden = [
    "Estrategia", 
    "Riqueza Final ($)", 
    "Retorno Total Histórico (%)",
    "Sharpe Ratio Histórico", 
    "Sortino Ratio Histórico", 
    "Max Drawdown (%)",
    "Volatilidad Anual Histórica (%)", 
    "Costo Total TC ($)", 
    "N° Rebalanceos"
]

df_resumen = pd.DataFrame(metricas_lista).rename(columns={
    "Retorno Total (%)": "Retorno Total Histórico (%)",
    "Sharpe Ratio": "Sharpe Ratio Histórico",
    "Sortino Ratio": "Sortino Ratio Histórico",
    "Volatilidad Anual (%)": "Volatilidad Anual Histórica (%)"
})[columnas_orden]

print("\n--- RANKING POR SHARPE RATIO HISTÓRICO ---")
print(df_resumen.sort_values("Sharpe Ratio Histórico", ascending=False).to_string(index=False))

print("\n--- RANKING POR RIQUEZA FINAL ---")
print(df_resumen.sort_values("Riqueza Final ($)", ascending=False).to_string(index=False))


--- RANKING POR SHARPE RATIO HISTÓRICO ---
                             Estrategia  Riqueza Final ($)  Retorno Total Histórico (%)  Sharpe Ratio Histórico  Sortino Ratio Histórico  Max Drawdown (%)  Volatilidad Anual Histórica (%)  Costo Total TC ($)  N° Rebalanceos
     7. Programación Dinámica (Bellman)      256003.998178                   156.003998                0.477603                 0.943388        -39.774124                        27.694376          464.267892              22
      6. NSGA-II (Rebalanceado Mensual)      251612.064759                   151.612065                0.471687                 0.931071        -40.953613                        27.655792          848.150414             118
    4. Markowitz (Rebalanceado Mensual)      251091.558523                   151.091559                0.470615                 0.922945        -40.768411                        27.688569          865.582180             118
                5. NSGA-II (Buy & Hold)      208131.941123  

## 6. Visualizaciones Interactivas con Plotly

Generamos los gráficos interactivos para analizar y comparar la trayectoria histórica de la riqueza, drawdowns, perfil riesgo-retorno y la composición de los portafolios óptimos.

In [8]:
colores = ['#95a5a6', '#3498db', '#f1c40f', '#e67e22', '#9b59b6', '#34495e', '#2ecc71']

# 1. Gráfico superpuesto de Riqueza
fig_riqueza = go.Figure()
for idx, nombre in enumerate(nombres_estrategias):
    fig_riqueza.add_trace(go.Scatter(
        x=fechas_sim, y=W[idx],
        mode='lines',
        name=nombre,
        line=dict(color=colores[idx], width=3.5 if idx == 6 else 2)
))
fig_riqueza.update_layout(title="Evolución de Riqueza (7 Estrategias)", xaxis_title="Fecha", yaxis_title="Capital (USD)", hovermode="x unified", template='plotly_dark')
fig_riqueza.show()

# 2. Gráfico de Drawdowns
def calcular_drawdown_percent(riqueza_path):
    cum_max = np.maximum.accumulate(riqueza_path)
    with np.errstate(divide='ignore', invalid='ignore'):
        dd = (riqueza_path - cum_max) / cum_max
        dd = np.nan_to_num(dd, nan=0.0)
    return dd * 100

fig_dd = go.Figure()
for idx, nombre in enumerate(nombres_estrategias):
    fig_dd.add_trace(go.Scatter(
        x=fechas_sim, y=calcular_drawdown_percent(W[idx]),
        mode='lines',
        name=nombre,
        line=dict(color=colores[idx], width=2.5 if idx == 6 else 1.5)
    ))
fig_dd.update_layout(title="Drawdowns Históricos (7 Estrategias)", xaxis_title="Fecha", yaxis_title="Drawdown (%)", hovermode="x unified", template='plotly_dark')
fig_dd.show()

# 3. Perfil Riesgo-Retorno
fig_risk_return = px.scatter(
    df_resumen,
    x='Volatilidad Anual Histórica (%)',
    y='Retorno Total Histórico (%)',
    color='Sharpe Ratio Histórico',
    size='Riqueza Final ($)',
    text='Estrategia',
    title='Perfil Riesgo-Retorno de las Estrategias',
    color_continuous_scale='Viridis',
    template='plotly_dark'
)
fig_risk_return.update_traces(textposition='top center')
fig_risk_return.show()

# 4. Comparativa de Pesos por Método
df_pesos_comp = pd.DataFrame({
    'Ticker': TICKERS_ORDENADOS * 3,
    'Peso (%)': np.concatenate([w_markowitz * 100, w_nsga2 * 100, w_eq * 100]).round(2),
    'Método': ['Markowitz'] * len(TICKERS_ORDENADOS) + ['NSGA-II (GA)'] * len(TICKERS_ORDENADOS) + ['Equiponderado'] * len(TICKERS_ORDENADOS)
})
fig_pesos_comp = px.bar(
    df_pesos_comp,
    x='Ticker',
    y='Peso (%)',
    color='Método',
    barmode='group',
    title='Comparación de Pesos de Portafolio por Método',
    color_discrete_sequence=['#e74c3c', '#9b59b6', '#3498db'],
    template='plotly_dark'
)
fig_pesos_comp.show()

# 5. Gráficos de Barras Horizontales de Métricas Clave
fig_w = px.bar(df_resumen.sort_values("Riqueza Final ($)"), x="Riqueza Final ($)", y="Estrategia", orientation='h', title="Capital Final Alcanzado (USD)", color="Riqueza Final ($)", color_continuous_scale="Viridis", template='plotly_dark')
fig_w.show()

fig_s = px.bar(df_resumen.sort_values("Sharpe Ratio Histórico"), x="Sharpe Ratio Histórico", y="Estrategia", orientation='h', title="Relación Riesgo-Retorno (Sharpe Ratio Histórico)", color="Sharpe Ratio Histórico", color_continuous_scale="Cividis", template='plotly_dark')
fig_s.show()

fig_tc = px.bar(df_resumen.sort_values("Costo Total TC ($)"), x="Costo Total TC ($)", y="Estrategia", orientation='h', title="Costo Total Acumulado por Transacciones (USD)", color="Costo Total TC ($)", color_continuous_scale="Magma", template='plotly_dark')
fig_tc.show()

## 7. Exportación a Excel

Guardamos el resumen general, las tablas de ranking, y las trayectorias de riqueza de todas las estrategias en un libro de Excel en la carpeta `salidas`.

In [9]:
df_sharpe = df_resumen.sort_values("Sharpe Ratio Histórico", ascending=False).reset_index(drop=True)
df_wealth = df_resumen.sort_values("Riqueza Final ($)", ascending=False).reset_index(drop=True)

excel_path = "salidas/reporte_comparativo_portafolios.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_resumen.to_excel(writer, index=False, sheet_name='Comparativo_General')
    df_sharpe.to_excel(writer, index=False, sheet_name='Ranking_Sharpe')
    df_wealth.to_excel(writer, index=False, sheet_name='Ranking_Riqueza')
    pd.DataFrame({n: trayectorias[n] for n in nombres_estrategias}).to_excel(writer, index=False, sheet_name='Trayectorias')

print(f"Reporte comparativo completo exportado a {excel_path}.")

Reporte comparativo completo exportado a salidas/reporte_comparativo_portafolios.xlsx.
